In [1]:
!pip install -q streamlit "google-genai>=2.12.1,<3" "requests==2.32.4" pyngrok

In [2]:

%%writefile /content/StyleTrip_AI/app.py

import streamlit as st
import requests
import time
from datetime import datetime
from google import genai
from google.genai import types

# =========================================================
# STYLETRIP AI
# Generative AI-Based Travel Outfit Recommendation System
# =========================================================

st.set_page_config(
    page_title="StyleTrip AI",
    page_icon="🧳",
    layout="wide",
    initial_sidebar_state="expanded"
)

# -------------------- DESIGN --------------------

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700;800&display=swap');

html, body, [class*="css"] {
    font-family: 'Poppins', sans-serif;
}
.stApp {
    background: linear-gradient(135deg, #fff8f3 0%, #f7f0ff 50%, #f1f8ff 100%);
}
.hero {
    padding: 35px 30px;
    border-radius: 24px;
    background: linear-gradient(120deg, #49306e, #9468a9, #d69aa7);
    color: white;
    margin-bottom: 25px;
}
.hero h1 {
    font-size: 42px;
    font-weight: 800;
    margin-bottom: 5px;
}
.hero p {
    font-size: 16px;
    opacity: .95;
}
.section {
    background: rgba(255,255,255,.85);
    border: 1px solid #eadff0;
    border-radius: 18px;
    padding: 20px;
    margin: 12px 0;
}
.stButton button {
    background: linear-gradient(90deg, #68458b, #b477a1);
    color: white;
    border: none;
    border-radius: 12px;
    padding: 12px 22px;
    font-weight: 600;
}
.stButton button:hover {
    background: #49306e;
    color: white;
}
.small-note {
    color: #6e6079;
    font-size: 13px;
}
</style>
""", unsafe_allow_html=True)

# -------------------- HEADER --------------------

st.markdown("""
<div class="hero">
    <h1>🧳 StyleTrip AI</h1>
    <p>Your AI-powered personal travel fashion stylist.</p>
    <p>Plan smarter outfits, coordinate colours and pack confidently.</p>
</div>
""", unsafe_allow_html=True)

# -------------------- FUNCTIONS --------------------

def get_gemini_response(api_key, prompt):
    """Generate an outfit plan using Gemini with model fallback."""

    client = genai.Client(api_key=api_key)

    models = [
        "gemini-3.8-flash",
        "gemini-3.7-flash",
        "gemini-3.5-flash",
        "gemini-3-flash-preview"
    ]

    errors = []

    for model_name in models:
        for attempt in range(2):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0.8,
                        max_output_tokens=6000
                    )
                )

                if response.text:
                    return response.text, model_name

                errors.append(f"{model_name}: Empty response")

            except Exception as error:
                error_message = str(error)
                errors.append(f"{model_name}: {error_message[:250]}")

                # Retry temporary service/rate-limit errors
                if any(code in error_message for code in ["503", "429", "500"]):
                    time.sleep(2)
                else:
                    break

    raise RuntimeError(
        "Gemini could not generate a response. "
        + " | ".join(errors[-4:])
    )


def get_wikimedia_images(query, limit=6):
    """Retrieve real fashion reference images from Wikimedia Commons."""

    endpoint = "https://commons.wikimedia.org/w/api.php"

    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": query + " fashion clothing outfit",
        "gsrnamespace": 6,
        "gsrlimit": limit,
        "prop": "imageinfo",
        "iiprop": "url",
        "iiurlwidth": 700,
        "format": "json"
    }

    headers = {
        "User-Agent": "StyleTripAI/1.0 (travel fashion student project)"
    }

    try:
        response = requests.get(
            endpoint,
            params=params,
            headers=headers,
            timeout=20
        )
        response.raise_for_status()

        pages = response.json().get("query", {}).get("pages", {})

        images = []

        for page in pages.values():
            image_info = page.get("imageinfo", [{}])[0]
            image_url = image_info.get("thumburl") or image_info.get("url")

            if image_url:
                images.append({
                    "title": page.get("title", "Fashion reference"),
                    "url": image_url
                })

        return images

    except Exception:
        return []


def build_prompt(data):
    """Build a detailed, structured prompt for Gemini."""

    return f"""
You are StyleTrip AI, an expert international travel fashion stylist.

Create a detailed, practical, personalised travel outfit plan.

TRIP DETAILS
Destination: {data['destination']}
Duration: {data['days']} days
Travel purpose: {data['purpose']}
Weather: {data['weather']}
Activities: {', '.join(data['activities'])}

FASHION PREFERENCES
Preferred style: {', '.join(data['styles'])}
Colour preferences: {data['colours']}
Clothing preferences: {data['clothing']}
Avoid: {data['avoid']}
Budget: {data['budget']}
Additional requirements: {data['extra']}

IMPORTANT INSTRUCTIONS

1. Create a separate outfit recommendation for EVERY day.
2. Make the outfits appropriate for the destination and manually supplied weather.
3. Do not claim to have checked live weather.
4. Include top, bottom, footwear, accessories and optional outerwear.
5. Explain why the outfit is suitable for the day's activities.
6. Give practical colour combinations.
7. Avoid repeating the same complete outfit unnecessarily.
8. Respect the user's budget and clothing preferences.
9. Suggest comfortable travel clothing for journey days.
10. Include modest, practical alternatives when relevant.
11. Include a packing checklist with quantities.
12. Include a footwear checklist and accessories checklist.
13. Give useful fashion tips and packing advice.
14. Do not invent clothing items the user has said they cannot wear.
15. Make recommendations realistic and easy to follow.

OUTPUT FORMAT

# STYLETRIP AI — PERSONAL TRAVEL OUTFIT PLAN

## 1. Trip Style Overview
Summarise the overall styling theme.

## 2. Daily Outfit Planner
For each day, use this structure:

### Day 1 — [Theme]
- Outfit:
- Top:
- Bottom:
- Footwear:
- Accessories:
- Colour combination:
- Weather suitability:
- Activity suitability:
- Styling tip:

Continue for every day of the trip.

## 3. Complete Packing Checklist
Group into:
- Tops
- Bottoms
- Dresses / ethnic wear, if suitable
- Outerwear
- Footwear
- Accessories
- Essentials

Give suggested quantities.

## 4. Colour Coordination Guide
Suggest colours that work well together.

## 5. Budget-Friendly Styling Tips
Suggest ways to reuse pieces and avoid unnecessary purchases.

## 6. Final Travel Fashion Tips
Provide practical final advice.

Be specific, detailed, organised and useful.
"""


# -------------------- SIDEBAR --------------------

st.sidebar.title("⚙️ StyleTrip Settings")

api_key = st.sidebar.text_input(
    "Gemini API Key",
    type="password",
    placeholder="Enter your Gemini API key"
)

st.sidebar.caption(
    "Your API key is entered privately for this session. "
    "Do not share it with anyone."
)

st.sidebar.markdown("---")
st.sidebar.markdown("### ✨ About StyleTrip AI")
st.sidebar.write(
    "An AI-powered travel fashion recommendation system "
    "that creates personalised daily outfit plans."
)

st.sidebar.markdown("---")
st.sidebar.markdown("**Technology Stack**")
st.sidebar.write(
    "• Python\n"
    "• Streamlit\n"
    "• Google Gemini API\n"
    "• Wikimedia Commons API"
)

# -------------------- USER INPUT --------------------

st.markdown("## 🌍 Plan Your Travel Style")
st.write("Enter your trip details to receive a personalised outfit plan.")

col1, col2 = st.columns(2)

with col1:
    destination = st.text_input(
        "📍 Travel destination",
        placeholder="Example: Manali, Goa, Paris"
    )

    days = st.number_input(
        "📅 Number of travel days",
        min_value=1,
        max_value=30,
        value=3,
        step=1
    )

    purpose = st.selectbox(
        "🎯 Purpose of travel",
        [
            "Vacation",
            "Solo travel",
            "Adventure trip",
            "College trip",
            "Business trip",
            "Family trip",
            "Honeymoon",
            "Weekend getaway",
            "Other"
        ]
    )

    weather = st.selectbox(
        "🌤️ Expected weather",
        [
            "Hot and sunny",
            "Warm",
            "Humid",
            "Rainy",
            "Cold",
            "Very cold",
            "Windy",
            "Mixed weather"
        ]
    )

with col2:
    styles = st.multiselect(
        "👗 Preferred fashion styles",
        [
            "Korean",
            "Western",
            "Casual",
            "Streetwear",
            "Traditional",
            "Indo-Western",
            "Aesthetic",
            "Minimalist",
            "Bohemian",
            "Sporty",
            "Chic",
            "Elegant"
        ],
        default=["Casual", "Aesthetic"]
    )

    activities = st.multiselect(
        "🗺️ Planned activities",
        [
            "Sightseeing",
            "Trekking",
            "Beach",
            "Shopping",
            "Temple visits",
            "Adventure activities",
            "Photography",
            "Night outings",
            "Cafes",
            "Road trip",
            "Relaxation",
            "Formal events"
        ],
        default=["Sightseeing", "Photography"]
    )

    colours = st.text_input(
        "🎨 Preferred colours",
        placeholder="Example: Black, white, pastel pink"
    )

    budget = st.selectbox(
        "💰 Clothing budget",
        [
            "Use existing wardrobe only",
            "Under ₹1,000",
            "₹1,000 – ₹3,000",
            "₹3,000 – ₹5,000",
            "₹5,000 – ₹10,000",
            "Flexible budget"
        ]
    )

st.markdown("---")
st.markdown("## 💖 Personal Clothing Preferences")

col3, col4 = st.columns(2)

with col3:
    clothing = st.text_area(
        "Clothing preferences",
        placeholder=(
            "Example: Comfortable outfits, jeans, tops, "
            "dresses, oversized shirts..."
        ),
        height=110
    )

with col4:
    avoid = st.text_area(
        "Clothing to avoid",
        placeholder=(
            "Example: Heels, sleeveless tops, short dresses, "
            "tight clothing..."
        ),
        height=110
    )

extra = st.text_area(
    "📝 Additional requirements",
    placeholder=(
        "Tell us anything else: packing light, repeated clothing, "
        "specific dress codes, comfort needs, etc."
    )
)

# -------------------- GENERATE PLAN --------------------

st.markdown("---")

generate = st.button(
    "✨ Generate My Travel Outfit Plan",
    use_container_width=True
)

if generate:

    if not api_key.strip():
        st.error("Please enter your Gemini API key in the sidebar.")

    elif not destination.strip():
        st.error("Please enter your travel destination.")

    elif not styles:
        st.error("Please select at least one fashion style.")

    elif not activities:
        st.error("Please select at least one activity.")

    else:
        data = {
            "destination": destination.strip(),
            "days": int(days),
            "purpose": purpose,
            "weather": weather,
            "styles": styles,
            "activities": activities,
            "colours": colours.strip() or "No preference",
            "budget": budget,
            "clothing": clothing.strip() or "No specific preference",
            "avoid": avoid.strip() or "No restrictions specified",
            "extra": extra.strip() or "None"
        }

        prompt = build_prompt(data)

        with st.spinner(
            "✨ Your personal AI stylist is creating your travel outfits..."
        ):
            try:
                plan, used_model = get_gemini_response(
                    api_key.strip(),
                    prompt
                )

                st.session_state["styletrip_plan"] = plan
                st.session_state["styletrip_destination"] = destination
                st.session_state["styletrip_days"] = int(days)
                st.session_state["styletrip_model"] = used_model

            except Exception as error:
                st.error(f"Unable to generate the plan: {error}")

# -------------------- DISPLAY GENERATED PLAN --------------------

if "styletrip_plan" in st.session_state:

    st.markdown("---")
    st.markdown("# ✨ Your Personal Travel Outfit Plan")

    st.success(
        "Your outfit plan has been generated successfully!"
    )

    st.caption(
        "Generated using "
        + st.session_state.get("styletrip_model", "Gemini")
    )

    plan = st.session_state["styletrip_plan"]

    st.markdown(plan)

    # Download plan
    st.download_button(
        label="📥 Download My Outfit Plan",
        data=plan,
        file_name="StyleTrip_AI_Outfit_Plan.txt",
        mime="text/plain",
        use_container_width=True
    )

    # -------------------- IMAGE REFERENCES --------------------

    st.markdown("---")
    st.markdown("# 🖼️ Fashion Inspiration Gallery")

    st.write(
        "Explore real fashion reference images retrieved from "
        "Wikimedia Commons. These are reference photographs, "
        "not AI-generated images."
    )

    destination = st.session_state.get(
        "styletrip_destination", "travel"
    )

    image_query = st.text_input(
        "Search fashion inspiration",
        value=f"{destination} travel outfit"
    )

    if st.button("🔎 Find Fashion Images"):
        with st.spinner("Finding fashion reference images..."):
            images = get_wikimedia_images(image_query, limit=8)

        if images:
            cols = st.columns(4)

            for index, item in enumerate(images):
                with cols[index % 4]:
                    st.image(
                        item["url"],
                        use_container_width=True
                    )
                    st.caption(item["title"])

        else:
            st.info(
                "No images were found for this search. "
                "Try a simpler search term."
            )

    st.markdown("---")
    st.markdown("### 💜 Thank you for using StyleTrip AI")
    st.caption(
        "Plan your journey. Style your moments. Travel confidently."
    )

else:
    st.info(
        "Your personalised outfit recommendations will appear here "
        "after you generate your travel plan."
    )

# -------------------- FOOTER --------------------

st.markdown("---")
st.markdown(
    "<p style='text-align:center;color:#796b82;'>"
    "StyleTrip AI | Generative AI Travel Fashion Styling System"
    "</p>",
    unsafe_allow_html=True
)


Overwriting /content/StyleTrip_AI/app.py


In [3]:
import os

print("Project folder:", os.path.isdir("/content/StyleTrip_AI"))
print("App file:", os.path.isfile("/content/StyleTrip_AI/app.py"))
print("File size:", os.path.getsize("/content/StyleTrip_AI/app.py"), "bytes")

Project folder: True
App file: True
File size: 15070 bytes


In [4]:
import os

os.makedirs("/content/StyleTrip_AI", exist_ok=True)

print("Project folder ready!")

Project folder ready!


In [5]:
import os
import subprocess
import time
import requests

os.chdir("/content/StyleTrip_AI")

log_file = open("/content/StyleTrip_AI/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("Streamlit process running:", process.poll() is None)

if process.poll() is None:
    print("Application started on port 8501!")
else:
    print("Application failed to start. Check the log:")
    print(open("/content/StyleTrip_AI/streamlit.log").read()[-4000:])

Streamlit process running: True
Application started on port 8501!


In [6]:
!pkill -f "streamlit run app.py" || true

^C


In [11]:

import re

path = "/content/StyleTrip_AI/app.py"

with open(path, "r", encoding="utf-8") as f:
    code = f.read()

new_function = '''
def get_wikimedia_images(query, limit=8):
    """Search Wikimedia Commons and filter out unrelated documents."""

    endpoint = "https://commons.wikimedia.org/w/api.php"

    headers = {
        "User-Agent": "StyleTripAI/1.0 (student fashion project)"
    }

    excluded_words = [
        "book", "cover", "guide", "manual", "map",
        "poster", "document", "manuscript", "newspaper",
        "magazine cover", "catalogue", "catalog",
        "painting", "drawing", "logo", "flag",
        "screenshot", "scan", "archive"
    ]

    search_terms = [
        f'filetype:bitmap "{query}" fashion clothing',
        "filetype:bitmap women summer fashion outfit",
        "filetype:bitmap casual street style clothing"
    ]

    images = []
    seen_urls = set()

    for search_term in search_terms:
        params = {
            "action": "query",
            "generator": "search",
            "gsrsearch": search_term,
            "gsrnamespace": 6,
            "gsrlimit": 30,
            "prop": "imageinfo",
            "iiprop": "url|size|mime",
            "iiurlwidth": 700,
            "format": "json"
        }

        try:
            response = requests.get(
                endpoint,
                params=params,
                headers=headers,
                timeout=20
            )
            response.raise_for_status()

            pages = response.json().get(
                "query", {}
            ).get("pages", {})

            for page in pages.values():
                title = page.get("title", "")
                title_lower = title.lower()

                if any(word in title_lower for word in excluded_words):
                    continue

                info = page.get("imageinfo", [{}])[0]
                mime = info.get("mime", "")

                if not mime.startswith("image/"):
                    continue

                if info.get("width", 0) < 300:
                    continue

                image_url = info.get("thumburl") or info.get("url")

                if not image_url or image_url in seen_urls:
                    continue

                seen_urls.add(image_url)

                images.append({
                    "title": title,
                    "url": image_url
                })

                if len(images) >= limit:
                    return images

        except Exception as error:
            print("Wikimedia search error:", str(error))

    return images
'''

pattern = r"def get_wikimedia_images\(query, limit=6\):.*?(?=\ndef build_prompt)"

updated_code, count = re.subn(
    pattern,
    new_function.strip() + "\n\n",
    code,
    flags=re.DOTALL
)

if count == 1:
    # Change the default search away from the destination name
    updated_code = updated_code.replace(
        'value=f"{destination} travel outfit"',
        'value="women summer casual fashion outfit"'
    )

    with open(path, "w", encoding="utf-8") as f:
        f.write(updated_code)

    print("SUCCESS: Fashion image search updated!")
else:
    print("Could not find the existing image-search function.")


SUCCESS: Fashion image search updated!


In [12]:
import os
import subprocess
import time

os.chdir("/content/StyleTrip_AI")

# Stop the previous Streamlit process
subprocess.run(
    ["pkill", "-f", "streamlit run app.py"],
    capture_output=True
)

time.sleep(2)

log_file = open("/content/StyleTrip_AI/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(6)

print("Streamlit running:", process.poll() is None)

Streamlit running: True


In [9]:
print(open("/content/StyleTrip_AI/streamlit.log").read()[-5000:])


In [14]:

path = "/content/StyleTrip_AI/app.py"

with open(path, "r", encoding="utf-8") as f:
    code = f.read()

# Add a quick summary section to the Gemini prompt
old = """## 2. Daily Outfit Planner
For each day, use this structure:"""

new = """## 2. QUICK OUTFIT SUMMARY
First provide a compact Markdown table with one row per trip day.
Columns:
Day | Outfit Name | Main Clothing | Footwear | Accessories | Colour Combination

Keep each row short, attractive and easy to scan.

## 3. DETAILED DAILY OUTFIT PLANNER
For each day, use this structure:"""

if old in code:
    code = code.replace(old, new, 1)

# Update subsequent prompt section numbering
code = code.replace(
    "## 3. Complete Packing Checklist",
    "## 4. Complete Packing Checklist"
)
code = code.replace(
    "## 4. Colour Coordination Guide",
    "## 5. Colour Coordination Guide"
)
code = code.replace(
    "## 5. Budget-Friendly Styling Tips",
    "## 6. Budget-Friendly Styling Tips"
)
code = code.replace(
    "## 6. Final Travel Fashion Tips",
    "## 7. Final Travel Fashion Tips"
)

# Remove the complete image gallery from the application
start = code.find("# -------------------- IMAGE REFERENCES --------------------")
end = code.find("# -------------------- FOOTER --------------------")

if start != -1 and end != -1 and end > start:
    code = code[:start] + code[end:]
    print("Fashion image gallery removed!")
else:
    print("Gallery section not found; checking existing code.")

with open(path, "w", encoding="utf-8") as f:
    f.write(code)

print("StyleTrip AI updated successfully!")
print("Quick Outfit Summary added to the Gemini prompt.")


Fashion image gallery removed!
StyleTrip AI updated successfully!
Quick Outfit Summary added to the Gemini prompt.


In [15]:
from pyngrok import ngrok
from IPython.display import display, HTML

# Check existing tunnels
tunnels = ngrok.get_tunnels()

if tunnels:
    public_url = tunnels[0].public_url
else:
    tunnel = ngrok.connect(8501, "http")
    public_url = tunnel.public_url

print("YOUR STYLETRIP AI LINK:")
print(public_url)

display(HTML(
    f'<h3><a href="{public_url}" target="_blank">'
    '🚀 CLICK HERE TO OPEN STYLETRIP AI</a></h3>'
))

YOUR STYLETRIP AI LINK:
https://repent-unsterile-factoid.ngrok-free.dev
